In [1]:
import pandas as pd
import numpy as np
import os


In [2]:
excel_file_name = 'online_retail_II.xlsx'
project_dir = r'E:\github\c_online_retail_II'
file_path = os.path.join(project_dir, excel_file_name)

df = pd.read_excel(file_path)
df.head()
 

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# Remove cancelled invoices
df = df[~df['Invoice'].astype(str).str.contains('C', na=False)]

# Drop missing values
df = df.dropna()

# Keep only UK customers
df = df[df['Country'] == 'United Kingdom']

# Convert date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Remove returns or invalid prices
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# Calculate total price
df['TotalPrice'] = df['Quantity'] * df['Price']

print(f"Dataset shape after cleaning: {df.shape}")
df.head()


Dataset shape after cleaning: (370929, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [4]:
ref_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Reference date: {ref_date.date()}")


Reference date: 2010-12-10


In [5]:
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (ref_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print(f"RFM shape: {rfm.shape}")
rfm.describe()


RFM shape: (3969, 4)


,CustomerID,Recency,Frequency,Monetary
count,3969.000000,3969.000000,3969.000000,3969.000000
mean,15561.406148,91.094986,4.437390,1868.167287
std,1582.099836,97.071763,7.531933,7380.830030
min,12346.000000,1.000000,1.000000,2.950000
25%,14201.000000,18.000000,1.000000,304.200000
50%,15577.000000,52.000000,2.000000,669.910000
75%,16941.000000,136.000000,5.000000,1655.640000
max,18287.000000,374.000000,155.000000,349164.350000


In [6]:
sample_customer = rfm.iloc[0]['CustomerID']
print(f"Sample customer ID: {sample_customer}")
rfm[rfm['CustomerID'] == sample_customer]
 

Sample customer ID: 12346.0


,CustomerID,Recency,Frequency,Monetary
0,12346.0,165,11,372.86


In [7]:
total_revenue = rfm['Monetary'].sum()
pareto_cutoff = total_revenue * 0.8

rfm_sorted = rfm.sort_values('Monetary', ascending=False)
rfm_sorted['CumulativeRevenue'] = rfm_sorted['Monetary'].cumsum()

top_customers = rfm_sorted[rfm_sorted['CumulativeRevenue'] <= pareto_cutoff]

print(f"Total Revenue: £{total_revenue:,.2f}")
print(f"Customers contributing to 80% revenue: {len(top_customers)}")
print(f"Percentage of customers: {len(top_customers)/len(rfm)*100:.1f}%")
 

Total Revenue: £7,414,755.96
Customers contributing to 80% revenue: 1140
Percentage of customers: 28.7%


In [8]:
rfm_ranked = rfm.sort_values('Monetary', ascending=False)

top_20_count = int(len(rfm_ranked) * 0.2)
top_20_customers = rfm_ranked.head(top_20_count)

revenue_top_20 = top_20_customers['Monetary'].sum()
revenue_percentage = (revenue_top_20 / total_revenue) * 100

print(f"Top 20% customers count: {top_20_count}")
print(f"Revenue share: {revenue_percentage:.1f}%")
 

Top 20% customers count: 793
Revenue share: 71.9%


In [9]:
quantiles = rfm[['Recency', 'Frequency', 'Monetary']].quantile([0.25, 0.5, 0.75])
quantiles_dict = quantiles.to_dict()

quantiles, quantiles_dict
 

(      Recency  Frequency  Monetary
 0.25     18.0        1.0    304.20
 0.50     52.0        2.0    669.91
 0.75    136.0        5.0   1655.64,
 {'Recency': {0.25: 18.0, 0.5: 52.0, 0.75: 136.0},
  'Frequency': {0.25: 1.0, 0.5: 2.0, 0.75: 5.0},
  'Monetary': {0.25: 304.2, 0.5: 669.91, 0.75: 1655.64}})

In [10]:
def R_Score(x, p, d):
    # Recency: lower is better
    if x <= d[p][0.25]:
        return 4
    elif x <= d[p][0.50]:
        return 3
    elif x <= d[p][0.75]:
        return 2
    else:
        return 1

def FM_Score(x, p, d):
    # Frequency & Monetary: higher is better
    if x <= d[p][0.25]:
        return 1
    elif x <= d[p][0.50]:
        return 2
    elif x <= d[p][0.75]:
        return 3
    else:
        return 4


In [11]:
rfm_segmentation = rfm.copy()

rfm_segmentation['R_Quartile'] = rfm_segmentation['Recency'].apply(
    R_Score, args=('Recency', quantiles_dict)
)
rfm_segmentation['F_Quartile'] = rfm_segmentation['Frequency'].apply(
    FM_Score, args=('Frequency', quantiles_dict)
)
rfm_segmentation['M_Quartile'] = rfm_segmentation['Monetary'].apply(
    FM_Score, args=('Monetary', quantiles_dict)
)

rfm_segmentation.head()


,CustomerID,Recency,Frequency,Monetary,R_Quartile,F_Quartile,M_Quartile
0,12346.0,165,11,372.86,1,4,2
1,12608.0,40,1,415.79,3,1,2
2,12745.0,122,2,723.85,2,2,3
3,12746.0,176,1,254.55,1,1,1
4,12747.0,5,16,5080.53,4,4,4


In [12]:
rfm_segmentation['RFM_Score'] = (
    rfm_segmentation['R_Quartile'].astype(str) +
    rfm_segmentation['F_Quartile'].astype(str) +
    rfm_segmentation['M_Quartile'].astype(str)
)

rfm_segmentation[['RFM_Score']].value_counts().head()


RFM_Score
111          426
444          412
344          192
211          175
112          171
Name: count, dtype: int64

In [13]:
best_customers = rfm_segmentation[
    rfm_segmentation['RFM_Score'] == '444'
].sort_values('Monetary', ascending=False)

print(f"Best customers count (444): {len(best_customers)}")
print(f"Total revenue: £{best_customers['Monetary'].sum():,.2f}")

best_customers.head(10)


Best customers count (444): 412
Total revenue: £3,601,733.60


,CustomerID,Recency,Frequency,Monetary,R_Quartile,F_Quartile,M_Quartile,RFM_Score
3841,18102.0,1,89,349164.35,4,4,4,444
636,13694.0,9,94,131443.19,4,4,4,444
3403,17511.0,3,31,84541.17,4,4,4,444
1623,15061.0,3,86,83284.38,4,4,4,444
2790,16684.0,15,27,80489.21,4,4,4,444
2839,16754.0,8,29,65500.07,4,4,4,444
3723,17949.0,7,74,60117.60,4,4,4,444
205,13089.0,4,109,57912.03,4,4,4,444
1805,15311.0,1,121,56003.26,4,4,4,444
3364,17450.0,3,7,52422.30,4,4,4,444


In [14]:
print("Customer Segments Summary")
print("-" * 40)

segments = {
    'Best Customers (444)': '444',
    'Almost Lost (244)': '244',
    'Lost Customers (144)': '144',
    'Lost Cheap Customers (111)': '111'
}

for name, code in segments.items():
    count = len(rfm_segmentation[rfm_segmentation['RFM_Score'] == code])
    print(f"{name}: {count}")

print(f"Loyal Customers (F=4): {len(rfm_segmentation[rfm_segmentation['F_Quartile'] == 4])}")
print(f"Big Spenders (M=4): {len(rfm_segmentation[rfm_segmentation['M_Quartile'] == 4])}")


Customer Segments Summary
----------------------------------------
Best Customers (444): 412
Almost Lost (244): 98
Lost Customers (144): 17
Lost Cheap Customers (111): 426
Loyal Customers (F=4): 884
Big Spenders (M=4): 992


In [15]:
rfm_data = rfm[['Recency', 'Frequency', 'Monetary']].copy()
print(f"Clustering data shape: {rfm_data.shape}")
rfm_data.head()


Clustering data shape: (3969, 3)


,Recency,Frequency,Monetary
0,165,11,372.86
1,40,1,415.79
2,122,2,723.85
3,176,1,254.55
4,5,16,5080.53


In [16]:
rfm_log = pd.DataFrame({
    'Recency_log': np.log(rfm_data['Recency'] + 0.1),
    'Frequency_log': np.log(rfm_data['Frequency'] + 0.1),
    'Monetary_log': np.log(rfm_data['Monetary'] + 0.1)
})

rfm_log.head()


,Recency_log,Frequency_log,Monetary_log
0,5.106551,2.406945,5.921471
1,3.691376,0.095310,6.030421
2,4.804840,0.741937,6.584722
3,5.171052,0.095310,5.539890
4,1.629241,2.778819,8.533191


In [17]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

matrix = rfm_log.values
silhouette_scores = {}

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, n_init=50, random_state=42)
    labels = kmeans.fit_predict(matrix)
    silhouette_scores[k] = silhouette_score(matrix, labels)
    print(f"k={k}: silhouette={silhouette_scores[k]:.4f}")

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\nBest k: {best_k}")


k=2: silhouette=0.4081
k=3: silhouette=0.3212
k=4: silhouette=0.3399
k=5: silhouette=0.3251
k=6: silhouette=0.3208
k=7: silhouette=0.3195
k=8: silhouette=0.3035
k=9: silhouette=0.2936
k=10: silhouette=0.2887

Best k: 2


In [18]:
kmeans = KMeans(n_clusters=best_k, n_init=100, random_state=42)
clusters = kmeans.fit_predict(matrix)

rfm_segmentation['Cluster'] = clusters

print(f"Final Silhouette Score: {silhouette_score(matrix, clusters):.4f}")
rfm_segmentation[['Cluster']].value_counts()


Final Silhouette Score: 0.4081


Cluster
0          2349
1          1620
Name: count, dtype: int64

In [19]:
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer
from sklearn.decomposition import PCA

scalers = {
    'StandardScaler': StandardScaler(),
    'RobustScaler': RobustScaler(),
    'PowerTransformer': PowerTransformer()
}

best_score = 0

for name, scaler in scalers.items():
    try:
        scaled = scaler.fit_transform(matrix)
        pca = PCA(n_components=3, random_state=42)
        pca_data = pca.fit_transform(scaled)

        labels = KMeans(n_clusters=best_k, n_init=50, random_state=42).fit_predict(pca_data)
        score = silhouette_score(pca_data, labels)

        print(f"{name} + PCA: silhouette={score:.4f}, explained_var={pca.explained_variance_ratio_.sum():.2f}")

        if score > best_score:
            best_score = score
            best_pca_data = pca_data
            best_clusters = labels
    except Exception as e:
        print(f"{name} failed: {e}")

print(f"\nBest PCA-based silhouette: {best_score:.4f}")


StandardScaler + PCA: silhouette=0.4233, explained_var=1.00
RobustScaler + PCA: silhouette=0.4168, explained_var=1.00
PowerTransformer + PCA: silhouette=0.4251, explained_var=1.00

Best PCA-based silhouette: 0.4251


In [20]:
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

algorithms = {
    'KMeans': KMeans(n_clusters=best_k, random_state=42),
    'GaussianMixture': GaussianMixture(n_components=best_k, random_state=42),
    'Agglomerative': AgglomerativeClustering(n_clusters=best_k),
    'DBSCAN': DBSCAN(eps=0.5, min_samples=5)
}

results = {}

for name, algo in algorithms.items():
    try:
        labels = algo.fit_predict(best_pca_data)
        if len(set(labels)) > 1:
            score = silhouette_score(best_pca_data, labels)
            results[name] = score
            print(f"{name}: silhouette={score:.4f}")
    except Exception as e:
        print(f"{name} error: {e}")

if results:
    best_algo = max(results, key=results.get)
    print(f"\nBest algorithm: {best_algo} ({results[best_algo]:.4f})")


KMeans: silhouette=0.4248
GaussianMixture: silhouette=0.4013
Agglomerative: silhouette=0.4040
DBSCAN: silhouette=0.3933

Best algorithm: KMeans (0.4248)


In [21]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

iso = IsolationForest(contamination=0.05, random_state=42)
labels = iso.fit_predict(matrix)

clean_matrix = matrix[labels == 1]

print(f"Outliers removed: {len(matrix) - len(clean_matrix)}")

kmeans_clean = KMeans(n_clusters=best_k, random_state=42, n_init=50)
clean_clusters = kmeans_clean.fit_predict(clean_matrix)

print(f"Silhouette (clean data): {silhouette_score(clean_matrix, clean_clusters):.4f}")


Outliers removed: 199
Silhouette (clean data): 0.4048


In [22]:
rfm_df = rfm.copy()
rfm_df['Cluster'] = best_clusters
rfm_df.head()


,CustomerID,Recency,Frequency,Monetary,Cluster
0,12346.0,165,11,372.86,0
1,12608.0,40,1,415.79,1
2,12745.0,122,2,723.85,1
3,12746.0,176,1,254.55,1
4,12747.0,5,16,5080.53,0


In [23]:
cluster_summary = rfm_df.groupby('Cluster').agg(
    Customer_Count=('Monetary', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).round(2)

cluster_summary['Revenue_Share_%'] = (
    cluster_summary['Total_Revenue'] / cluster_summary['Total_Revenue'].sum() * 100
).round(1)

cluster_summary


,Customer_Count,Avg_Recency,Avg_Frequency,Avg_Monetary,Total_Revenue,Revenue_Share_%
Cluster,,,,,,
0,1992,37.92,7.41,3335.78,6644877.28,89.6
1,1977,144.67,1.44,389.42,769878.68,10.4


In [24]:
segment_info = {}

for cluster, row in cluster_summary.iterrows():
    if row.Avg_Recency < 60 and row.Avg_Frequency > 10:
        name = "VIP Customers"
        strategy = "Loyalty program, exclusive offers, priority service"
    elif row.Avg_Recency < 120 and row.Avg_Frequency > 5:
        name = "Loyal Customers"
        strategy = "Personalized offers, email campaigns"
    elif row.Avg_Recency > 200:
        name = "At-Risk Customers"
        strategy = "Win-back campaign, discounts, reminders"
    elif row.Avg_Frequency < 3:
        name = "New Customers"
        strategy = "Onboarding, first repeat purchase incentives"
    else:
        name = "Regular Customers"
        strategy = "Cross-sell & upsell strategies"

    segment_info[cluster] = {
        'Segment': name,
        'Strategy': strategy
    }

segment_info


{0: {'Segment': 'Loyal Customers',
  'Strategy': 'Personalized offers, email campaigns'},
 1: {'Segment': 'New Customers',
  'Strategy': 'Onboarding, first repeat purchase incentives'}}

In [25]:
rfm_df['Segment'] = rfm_df['Cluster'].map(lambda x: segment_info[x]['Segment'])
rfm_df['Strategy'] = rfm_df['Cluster'].map(lambda x: segment_info[x]['Strategy'])

rfm_df[['Segment', 'Strategy']].head()


,Segment,Strategy
0,Loyal Customers,"Personalized offers, email campaigns"
1,New Customers,"Onboarding, first repeat purchase incentives"
2,New Customers,"Onboarding, first repeat purchase incentives"
3,New Customers,"Onboarding, first repeat purchase incentives"
4,Loyal Customers,"Personalized offers, email campaigns"


In [26]:
from datetime import datetime

def executive_summary(rfm_df, segment_info):
    total_customers = len(rfm_df)
    total_revenue = rfm_df['Monetary'].sum()

    summary = f"""
Customer Segmentation Executive Report
Generated on: {datetime.now().strftime('%Y-%m-%d')}

Total Customers: {total_customers:,}
Total Revenue: £{total_revenue:,.0f}

Segment Overview:
"""

    for cluster, info in segment_info.items():
        segment_data = rfm_df[rfm_df['Cluster'] == cluster]
        revenue = segment_data['Monetary'].sum()

        summary += f"""
- {info['Segment']}
  Customers: {len(segment_data):,}
  Revenue Share: {revenue / total_revenue * 100:.1f}%
  Strategy: {info['Strategy']}
"""

    return summary

report = executive_summary(rfm_df, segment_info)
print(report)



Customer Segmentation Executive Report
Generated on: 2026-02-12

Total Customers: 3,969
Total Revenue: £7,414,756

Segment Overview:

- Loyal Customers
  Customers: 1,992
  Revenue Share: 89.6%
  Strategy: Personalized offers, email campaigns

- New Customers
  Customers: 1,977
  Revenue Share: 10.4%
  Strategy: Onboarding, first repeat purchase incentives



In [27]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

rfm_scored = rfm_df.copy()
rfm_scored['R_Score'] = 100 - scaler.fit_transform(rfm_scored[['Recency']])
rfm_scored['F_Score'] = scaler.fit_transform(rfm_scored[['Frequency']])
rfm_scored['M_Score'] = scaler.fit_transform(rfm_scored[['Monetary']])

rfm_scored['Total_Score'] = (
    0.3 * rfm_scored['R_Score'] +
    0.4 * rfm_scored['F_Score'] +
    0.3 * rfm_scored['M_Score']
)

rfm_scored['Tier'] = pd.cut(
    rfm_scored['Total_Score'],
    bins=[0, 40, 60, 80, 100],
    labels=['Bronze', 'Silver', 'Gold', 'Platinum']
)

rfm_scored[['Total_Score', 'Tier']].head()


,Total_Score,Tier
0,29.894388,Bronze
1,29.968987,Bronze
2,29.905898,Bronze
3,29.859466,Bronze
4,30.040107,Bronze


In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df_churn = rfm_df.copy()
df_churn['Churn_Risk'] = (
    (df_churn['Recency'] > 120) &
    (df_churn['Frequency'] < df_churn['Frequency'].median())
).astype(int)

X = df_churn[['Recency', 'Frequency', 'Monetary']]
y = df_churn['Churn_Risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

df_churn['Churn_Probability'] = model.predict_proba(X)[:, 1]

df_churn[['Churn_Probability']].describe()


,Churn_Probability
count,3969.000000
mean,0.167080
std,0.370578
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


In [29]:
clv = rfm_df.copy()
clv['CLV'] = clv['Monetary'] * 0.3  # gross margin assumption

clv['CLV_Level'] = pd.qcut(
    clv['CLV'],
    q=4,
    labels=['Low', 'Medium', 'High', 'Very High']
)

clv[['CLV', 'CLV_Level']].head()


,CLV,CLV_Level
0,111.858,Medium
1,124.737,Medium
2,217.155,High
3,76.365,Low
4,1524.159,Very High


In [30]:
output_dir = r'E:\github\c_online_retail_II\output'
os.makedirs(output_dir, exist_ok=True)

final_output = rfm_df.copy()

final_output = final_output.merge(
    rfm_scored[['Total_Score', 'Tier']],
    left_index=True,
    right_index=True
)

final_output = final_output.merge(
    df_churn[['Churn_Probability']],
    left_index=True,
    right_index=True
)

final_output = final_output.merge(
    clv[['CLV', 'CLV_Level']],
    left_index=True,
    right_index=True
)

final_output.reset_index(inplace=True)

file_path = os.path.join(output_dir, 'customer_segmentation_final.xlsx')
final_output.to_excel(file_path, index=False)

print(f"✅ File saved successfully at:\n{file_path}")


✅ File saved successfully at:
E:\github\c_online_retail_II\output\customer_segmentation_final.xlsx
